In [49]:
# imports
from importlib.metadata import version
from pathlib import Path
import json
import math
import random
import sys
import time

import tokenizers
from datasets import load_dataset
from IPython.display import display
from tqdm.auto import tqdm
from tokenizers import Tokenizer, models
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.processors import TemplateProcessing
from tokenizers.trainers import BpeTrainer

In [50]:
# config
TOKENIZERS_VERSION = "0.22.1"
HF_DATASET = "Similoluwa/african-multilingual-tokenizer-challenge"
HF_REVISION = "v1.0.0"
LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
MAX_VOCAB_SIZE = 10_000
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>"]
SUBMISSION_PATH = "tokenizer.json"
RNG_SEED = 42
LANGUAGE_REPEAT = {"en": 1, "fr": 2.5, "ha": 5, "sw": 5, "yo": 4, "am": 5.5}
TRANSITION_RATIO = 0.85
STAGE_ONE_SIZE = int(MAX_VOCAB_SIZE * TRANSITION_RATIO)
SYMBOL_BASE = 0xF0000
MERGE_BUFFER = 32


In [51]:
# check tokenizers version
installed_version = version("tokenizers")
if installed_version != TOKENIZERS_VERSION:
    raise RuntimeError(
        f"This challenge requires tokenizers=={TOKENIZERS_VERSION}; "
        f"found {installed_version}."
    )

print("tokenizers version:", tokenizers.__version__)
print("Maximum vocabulary:", MAX_VOCAB_SIZE)

tokenizers version: 0.22.1
Maximum vocabulary: 10000


In [52]:
# load a dataset split
def load_competition_data(split):
    """Load one split and return its language/text columns."""
    if split not in {"train", "validation"}:
        raise ValueError("split must be 'train' or 'validation'")
    dataset = load_dataset(HF_DATASET, split=split, revision=HF_REVISION)
    frame = dataset.to_pandas()[["language", "text"]].copy()
    frame["language"] = frame["language"].str.lower().str.strip()
    return frame.reset_index(drop=True)

In [53]:
# load splits and check languages match
train = load_competition_data("train")
validation = load_competition_data("validation")
assert set(train.language) == set(validation.language) == set(LANGUAGES), "missing languages"

In [54]:
# row/char counts per language
print(f"Train rows: {len(train):,}")
print(f"Validation rows: {len(validation):,}")
display(train.groupby("language").agg(
    rows=("text", "size"),
    characters=("text", lambda texts: texts.str.len().sum()),
))

Train rows: 240,000
Validation rows: 24,000


,rows,characters
language,,
am,40000,4079673
en,40000,5429908
fr,40000,5686404
ha,40000,5251977
sw,40000,4376925
yo,40000,4629899


In [55]:
# upsample each language by its repeat factor
def build_weighted_corpus(frame, repeat_factors, seed):
    """Repeat each language's texts by its weight."""
    rng = random.Random(seed)
    corpus = []
    for language in LANGUAGES:
        texts = [text for text in frame.loc[frame.language == language, "text"] if text]
        factor = repeat_factors[language]
        whole = math.floor(factor)
        corpus.extend(texts * whole)
        remainder = factor - whole
        if remainder > 0:
            corpus.extend(rng.sample(texts, round(len(texts) * remainder)))
    return corpus

In [56]:
# iterate corpus with a progress bar
def iter_training_sentences(corpus):
    """Yield sentences from corpus with a progress bar."""
    yield from tqdm(corpus, desc="Streaming training sentences", unit="sentences", dynamic_ncols=True)

In [57]:
# build weighted corpus
weighted_corpus = build_weighted_corpus(train, LANGUAGE_REPEAT, RNG_SEED)

print("Language repeat factors:", LANGUAGE_REPEAT)
print("Weighted training sentences:", f"{len(weighted_corpus):,}")

Language repeat factors: {'en': 1, 'fr': 2.5, 'ha': 5, 'sw': 5, 'yo': 4, 'am': 5.5}
Weighted training sentences: 920,000


In [58]:
# stage-one tokenizer
stage_one = Tokenizer(models.BPE())
stage_one.pre_tokenizer = ByteLevel(add_prefix_space=False, use_regex=True)
stage_one.decoder = ByteLevelDecoder()

In [59]:
# stage-one trainer config
stage_one_trainer = BpeTrainer(
    vocab_size=STAGE_ONE_SIZE,
    special_tokens=SPECIAL_TOKENS,
    initial_alphabet=ByteLevel.alphabet(),
    show_progress=True,
)

In [60]:
# train stage one
started = time.perf_counter()
stage_one.train_from_iterator(
    iter_training_sentences(weighted_corpus),
    trainer=stage_one_trainer,
    length=len(weighted_corpus),
)
stage_one_seconds = time.perf_counter() - started

print(f"Stage one time: {stage_one_seconds:.2f} seconds")
print(f"Stage one vocabulary: {stage_one.get_vocab_size():,} / {STAGE_ONE_SIZE:,}")

Streaming training sentences:   0%|           | 0/920000 [00:00<?, ?sentences/s]




Stage one time: 81.92 seconds
Stage one vocabulary: 8,500 / 8,500


In [61]:
# encode each text as a string of stage-one token symbols
def to_symbol_corpus(stage_one, corpus, symbol_base):
    """Represent each stage-one token id as one private-use character."""
    unique_texts = list(dict.fromkeys(corpus))
    encodings = stage_one.encode_batch(unique_texts, add_special_tokens=False)
    symbols = {
        text: "".join(chr(symbol_base + token_id) for token_id in encoding.ids)
        for text, encoding in zip(unique_texts, encodings)
    }
    return [symbols[text] for text in corpus]

In [62]:
# sanity checks, then build symbol corpus
assert [stage_one.token_to_id(token) for token in SPECIAL_TOKENS] == list(range(len(SPECIAL_TOKENS)))
assert SYMBOL_BASE + stage_one.get_vocab_size() <= 0xFFFFD

symbol_corpus = to_symbol_corpus(stage_one, weighted_corpus, SYMBOL_BASE)
print("Symbol sentences:", f"{len(symbol_corpus):,}")

Symbol sentences: 920,000


In [63]:
# stage-two trainer 
symbol_alphabet = [
    chr(SYMBOL_BASE + token_id)
    for token_id in range(len(SPECIAL_TOKENS), stage_one.get_vocab_size())
]

stage_two = Tokenizer(models.BPE())
stage_two_trainer = BpeTrainer(
    vocab_size=MAX_VOCAB_SIZE - len(SPECIAL_TOKENS) + MERGE_BUFFER,
    initial_alphabet=symbol_alphabet,
    show_progress=True,
)

In [64]:
# train stage two
started = time.perf_counter()
stage_two.train_from_iterator(
    iter_training_sentences(symbol_corpus),
    trainer=stage_two_trainer,
    length=len(symbol_corpus),
)
stage_two_seconds = time.perf_counter() - started

print(f"Stage two time: {stage_two_seconds:.2f} seconds")
print(f"Stage two merges: {len(json.loads(stage_two.to_str())['model']['merges']):,}")

Streaming training sentences:   0%|           | 0/920000 [00:00<?, ?sentences/s]




Stage two time: 55.58 seconds
Stage two merges: 1,532


In [65]:

def assemble_superbpe(stage_one, stage_two, symbol_base, target_vocab_size):
    """Merge stage-one and stage-two vocab/merges, stopping once vocab hits target_vocab_size."""
    first = json.loads(stage_one.to_str())["model"]
    second = json.loads(stage_two.to_str())["model"]
    token_by_id = {index: token for token, index in first["vocab"].items()}

    def expand(symbols):
        return "".join(token_by_id[ord(symbol) - symbol_base] for symbol in symbols)

    vocab = dict(first["vocab"])
    merges = [tuple(pair) for pair in first["merges"]]
    for left, right in second["merges"]:
        left_token, right_token = expand(left), expand(right)
        merges.append((left_token, right_token))
        vocab.setdefault(left_token + right_token, len(vocab))
        if len(vocab) == target_vocab_size:
            break
    if len(vocab) != target_vocab_size:
        raise RuntimeError(f"only reached {len(vocab)} / {target_vocab_size} tokens; raise MERGE_BUFFER")
    return vocab, merges

In [66]:
# assemble to exactly MAX_VOCAB_SIZE, build tokenizer
vocab, merges = assemble_superbpe(stage_one, stage_two, SYMBOL_BASE, MAX_VOCAB_SIZE)

tokenizer = Tokenizer(models.BPE(vocab=vocab, merges=merges))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False, use_regex=False)
tokenizer.decoder = ByteLevelDecoder()
tokenizer.add_special_tokens(SPECIAL_TOKENS)

3

In [67]:
# bos/eos template
tokenizer.post_processor = TemplateProcessing(
    single="<bos> $A <eos>",
    pair="<bos> $A <eos> <bos> $B <eos>",
    special_tokens=[
        ("<bos>", tokenizer.token_to_id("<bos>")),
        ("<eos>", tokenizer.token_to_id("<eos>")),
    ],
)

In [68]:
# validate final tokenizer
vocab_size = tokenizer.get_vocab_size(with_added_tokens=True)
assert vocab_size == MAX_VOCAB_SIZE, (vocab_size, MAX_VOCAB_SIZE)
assert tokenizer.token_to_id("<unk>") is None
assert set(ByteLevel.alphabet()) <= set(tokenizer.get_vocab()), "byte alphabet incomplete"
assert [tokenizer.token_to_id(token) for token in SPECIAL_TOKENS] == list(range(len(SPECIAL_TOKENS)))

In [69]:
# save tokenizer
tokenizer.save(SUBMISSION_PATH, pretty=True)
print(f"Saved {SUBMISSION_PATH} with {vocab_size:,} / {MAX_VOCAB_SIZE:,} tokens")

Saved tokenizer.json with 10,000 / 10,000 tokens


In [70]:
# find utils.py and add it to sys.path
def ensure_utils():
    """Locate utils.py and add its folder to sys.path."""
    for root in (Path.cwd(), *Path.cwd().parents):
        for relative in ("utils.py", "starter/utils.py"):
            candidate = root / relative
            if candidate.is_file():
                location = str(candidate.parent)
                if location not in sys.path:
                    sys.path.insert(0, location)
                return
    raise FileNotFoundError("Place the supplied competition utils.py beside this notebook.")

In [71]:
# run official validation
ensure_utils()
from utils import profile_submission

validation_report = profile_submission(SUBMISSION_PATH, data=validation, repeats=3)
assert validation_report["valid"], validation_report["errors"]
assert validation_report["lossy_rows"] == 0
print(f"Official validation score: {validation_report['score']:.4f}")

AI Research Foundations Multilingual Tokenization Challenge
Submission checker

Loading tokenizer......... ✓
File size................. ✓
Vocabulary................ ✓ 10,000 / 10,000
Encoding.................. ✓
Compatibility............. ✓

Scores (24,000 rows, lower is better)
language      tokens/word  [UNK] rate    score
English             2.054      0.0000    2.054
French              2.025      0.0000    2.025
Hausa*              1.582      0.0000    1.582
Swahili*            1.755      0.0000    1.755
Yoruba*             1.859      0.0000    1.859
Amharic*            2.351      0.0000    2.351
Guardrail penalty......... 0.0000
Reconstruction penalty.... 0.0000
SCORE..................... 1.8868
  * scored languages
  guardrail 2.170, highest is English at 2.054 (5.4% headroom)
  reconstruction 100.0%, charged 3 x the share not reconstructed

Local benchmark (informational only)
Evaluation time........... 1.59 s
Throughput................ 1.9M characters/sec

READY FOR SUBMISSION

In [72]:
# reload from disk as a final check
saved_tokenizer = Tokenizer.from_file(SUBMISSION_PATH)
assert saved_tokenizer.get_vocab_size(with_added_tokens=True) == MAX_VOCAB_SIZE

print("Final local score:", f"{validation_report['score']:.4f}")

Final local score: 1.8868
